In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

from pyspark.sql.window import Window

In [3]:
spark = SparkSession.builder.appName("AnalyseProcessed").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 13:52:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.sparkContext.setLogLevel("ERROR")

In [5]:
from pathlib import Path

notebook_dir = Path().resolve()  # notebooks/
data_path = notebook_dir.parent / "data" / "processed"

df = spark.read.option("header", True).parquet(str(data_path))

df.show(7)

+---------+---------------------+--------------------+---------+---------------------+--------------------+-------------------+-------------+-----------------------+----------+--------------------+--------------------+----------+----------+---------------------+--------------------------------+---------------+----------+--------------------+--------------------+---------------------------+-------------------+------------+--------------------+--------------------+-------------------+------------+
|      ath|ath_change_percentage|            ath_date|      atl|atl_change_percentage|            atl_date| circulating_supply|current_price|fully_diluted_valuation|  high_24h|                  id|        last_updated|   low_24h|market_cap|market_cap_change_24h|market_cap_change_percentage_24h|market_cap_rank|max_supply|                name|    price_change_24h|price_change_percentage_24h|       total_supply|total_volume|              7d_avg|              7d_max|             7d_low|updated_date

The data below shows that some coins appear on more days than others which is a data quality issue. 

In [6]:
from pyspark.sql.functions import count

df.groupBy("updated_date").agg(count("name").alias("coin_count")).show()

+------------+----------+
|updated_date|coin_count|
+------------+----------+
|  2026-02-26|       101|
|  2026-03-05|       105|
|  2026-03-11|        99|
|  2026-03-03|       100|
|  2026-03-08|       100|
|  2026-03-06|        99|
|  2026-02-27|        99|
|  2026-03-10|        99|
|  2026-03-02|        98|
|  2026-03-09|        98|
|  2026-03-01|         1|
|  2026-02-28|         1|
+------------+----------+



In [ ]:
# window_spec = Window.partitionBy("id").orderBy(desc("updated_date"))

# most_updated_data = (
#     df.withColumn("rn", row_number().over(window_spec)).filter("rn = 1").drop("rn")
# )

max_date = df.select(F.max("updated_date")).collect()[0][0]

recent_data = df.filter(F.col("updated_date") == max_date).select(
    "name",
    "current_price",
    "market_cap",
    "updated_date",
    "total_volume",
    "market_cap_rank",
)

recent_data.show(5)

+--------+-------------+----------+------------+------------+---------------+
|    name|current_price|market_cap|updated_date|total_volume|market_cap_rank|
+--------+-------------+----------+------------+------------+---------------+
|    Kite|     0.266254| 479351378|  2026-03-11| 8.1938028E7|            100|
|Filecoin|     0.867601| 659423825|  2026-03-11|1.75715029E8|             85|
|Midnight|     0.052317| 868703726|  2026-03-11|   9649376.0|             72|
|​​Stable|    0.0277138| 576203356|  2026-03-11| 2.0167999E7|             92|
|MemeCore|         1.42|2472756003|  2026-03-11|   8238049.0|             38|
+--------+-------------+----------+------------+------------+---------------+
only showing top 5 rows


## Ranking current price

In [8]:
rank_price_spec = Window.partitionBy("updated_date").orderBy(desc("current_price"))

# curr_top_price = ranking_df.sort(desc("current_price")).show(10)

curr_top_price = recent_data.withColumn(
    "current_price_rank", rank().over(rank_price_spec)
)

curr_top_price.show(10)

+------------+-------------+-------------+------------+---------------+---------------+------------------+
|        name|current_price|   market_cap|updated_date|   total_volume|market_cap_rank|current_price_rank|
+------------+-------------+-------------+------------+---------------+---------------+------------------+
|     Bitcoin|      69953.0|1398820091191|  2026-03-11|5.4010222508E10|              1|                 1|
|    PAX Gold|       5196.5|   2597629121|  2026-03-11|   4.11137867E8|             36|                 2|
| Tether Gold|       5157.0|   2910870297|  2026-03-11|   6.52222225E8|             34|                 3|
|    Ethereum|      2024.25| 244259394937|  2026-03-11|2.1773748003E10|              2|                 4|
|         BNB|       643.06|  87681625844|  2026-03-11|   8.19362067E8|              4|                 5|
|Bitcoin Cash|       451.39|   9030293297|  2026-03-11|   2.31927821E8|             14|                 6|
|      Monero|       357.94|   660299

## Ranking Market Cap

In [9]:
rank_market_spec = Window.partitionBy("updated_date").orderBy(desc("market_cap"))


curr_top_market = recent_data.withColumn(
    "current_market_cap_rank", rank().over(rank_market_spec)
)

curr_top_market.show(10)

+------------+-------------+-------------+------------+---------------+---------------+-----------------------+
|        name|current_price|   market_cap|updated_date|   total_volume|market_cap_rank|current_market_cap_rank|
+------------+-------------+-------------+------------+---------------+---------------+-----------------------+
|     Bitcoin|      69953.0|1398820091191|  2026-03-11|5.4010222508E10|              1|                      1|
|    Ethereum|      2024.25| 244259394937|  2026-03-11|2.1773748003E10|              2|                      2|
|      Tether|     0.999985| 183890371994|  2026-03-11|8.2331560771E10|              3|                      3|
|         BNB|       643.06|  87681625844|  2026-03-11|   8.19362067E8|              4|                      4|
|         XRP|         1.38|  84587944044|  2026-03-11|   3.50813802E9|              5|                      5|
|        USDC|     0.999967|  78664642327|  2026-03-11|1.3288489493E10|              6|                 

## Aggregate and Average Market Capitalization

In [10]:
df.select(
    "name", "current_price", "market_cap", "total_volume", "updated_date"
).createOrReplaceTempView("crypto_prices")

In [11]:
agg_market = spark.sql(
    """
    SELECT updated_date, sum(market_cap) as total_market_cap
    FROM crypto_prices 
    GROUP BY updated_date
    ORDER BY updated_date DESC
"""
)

agg_market.show()

+------------+----------------+
|updated_date|total_market_cap|
+------------+----------------+
|  2026-03-11|   2398267403533|
|  2026-03-10|   2399896922025|
|  2026-03-09|   2323232252261|
|  2026-03-08|   2332237691268|
|  2026-03-06|   2421682587777|
|  2026-03-05|   2481899981856|
|  2026-03-03|   2350972043210|
|  2026-03-02|   2295054269100|
|  2026-03-01|     15908254518|
|  2026-02-28|      1883793196|
|  2026-02-27|   2354599981906|
|  2026-02-26|   2382486193657|
+------------+----------------+



In [12]:
avg_market = spark.sql(
    """
    with avg_market as (
        SELECT name, avg(market_cap) as avg_market_cap
        FROM crypto_prices 
        GROUP BY name
    )
    
    SELECT * , RANK() 
    OVER (ORDER BY avg_market_cap DESC) as avg_market_cap_rank
    FROM avg_market
    
    
    """
)

avg_market.show()

+-------------+------------------+-------------------+
|         name|    avg_market_cap|avg_market_cap_rank|
+-------------+------------------+-------------------+
|      Bitcoin|1.3776601759463E12|                  1|
|     Ethereum| 2.447566175596E11|                  2|
|       Tether| 1.838006332515E11|                  3|
|          BNB|  8.65097640853E10|                  4|
|          XRP|  8.48326815881E10|                  5|
|         USDC|  7.67177336331E10|                  6|
|       Solana|  4.92225953663E10|                  7|
|         TRON|  2.70234659584E10|                  8|
| Figure Heloc|  1.60120901015E10|                  9|
|     Dogecoin|  1.49753924348E10|                 10|
|WhiteBIT Coin|  1.12168784407E10|                 11|
|         USDS|  1.05632493469E10|                 12|
|      Cardano|     9.965090589E9|                 13|
| Bitcoin Cash|     9.189795628E9|                 14|
|    LEO Token|    8.3094759762E9|                 15|
|  Hyperli

## Average Price

In [17]:
avg_price = spark.sql(
    """
    WITH average_prices AS (
        SELECT name, ROUND(avg(current_price), 3) as average_price
        FROM crypto_prices 
        GROUP BY name
        ORDER BY avg(current_price) DESC
    )
    SELECT *, RANK() OVER (ORDER BY average_price DESC) as avg_price_rank
    FROM average_prices
    """
)

avg_price.show()

+--------------------+-------------+--------------+
|                name|average_price|avg_price_rank|
+--------------------+-------------+--------------+
|             Bitcoin|      68839.2|             1|
|            PAX Gold|     5214.701|             2|
|         Tether Gold|     5173.367|             3|
|            Ethereum|     2026.751|             4|
|                 BNB|      634.144|             5|
|        Bitcoin Cash|      458.937|             6|
|              Monero|      350.191|             7|
|               Zcash|      222.877|             8|
|           Bittensor|      186.978|             9|
|                OUSG|      114.479|            10|
|                Aave|      113.903|            11|
|                 OKB|       87.571|            12|
|              Solana|       86.283|            13|
|               Quant|       63.922|            14|
|            Litecoin|       54.665|            15|
|       WhiteBIT Coin|       52.538|            16|
|         Hy

## Volume to Market Ratio

In [14]:
curr_top_market = recent_data.withColumn("rank", rank().over(rank_market_spec))

vol_market_ratio = (
    df.filter((F.col("total_volume") != 0) & (F.col("market_cap") != 0))
    .withColumn(
        "vol_market_ratio",
        F.when(
            F.col("market_cap") > 0,
            F.round(F.col("total_volume") / F.col("market_cap"), 5),
        ).otherwise(0.0),
    )
    .orderBy(F.col("vol_market_ratio").desc())
    .select(
        "name",
        "current_price",
        "market_cap",
        "total_volume",
        "updated_date",
        "vol_market_ratio",
    )
)

vol_market_ratio.show(5)

rank_vol_market_spec = Window.orderBy(desc("vol_market_ratio"))
window_spec = Window.partitionBy("name").orderBy(desc("vol_market_ratio"))

# gets the highest vol-to-market ratio for each coin and ranks them
vol_market_ratio_rank = (
    vol_market_ratio.withColumn("rn", row_number().over(window_spec))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("vol_market_rank", rank().over(rank_vol_market_spec))
)

vol_market_ratio_rank.show(10)

+------+-------------+------------+----------------+------------+----------------+
|  name|current_price|  market_cap|    total_volume|updated_date|vol_market_ratio|
+------+-------------+------------+----------------+------------+----------------+
|  USD1|     0.999959|  4703542500|   4.050956274E9|  2026-02-27|         0.86126|
|  USD1|     0.999996|  4705551140|     3.7583704E9|  2026-02-26|         0.79871|
|Tether|          1.0|183912220526|1.11298241808E11|  2026-03-05|         0.60517|
|  USD1|     0.999117|  4615776647|   2.644099229E9|  2026-03-05|         0.57284|
|Tether|      0.99988|183606483628| 9.7552146231E10|  2026-03-03|         0.53131|
+------+-------------+------------+----------------+------------+----------------+
only showing top 5 rows
+-------------+-------------+------------+----------------+------------+----------------+---------------+
|         name|current_price|  market_cap|    total_volume|updated_date|vol_market_ratio|vol_market_rank|
+-------------+--

## Top Performing Assets 

find coins which meet the following criteria: 
- Market capitalization: the *higher* the *better* 
- Price: *higher* price means greater demand 
- Volume/Market Cap Ratio: determines stability of the coin. Dependent on other factors.

Therefore, the ranking is determined using weighted sums. 
1. Market cap contributes 40% → weight of `0.4`
2. Price contributes 40% → weight of `0.4`
3. Vol/market ratio contributes 20% → weight of `0.2`

In [18]:
top_performing_assets = avg_market.join(avg_price, on=["name"]).join(
    vol_market_ratio_rank.select(
        "name", "total_volume", "vol_market_ratio", "vol_market_rank"
    ),
    on=["name"],
)

top_performing_assets = top_performing_assets.withColumn(
    "top_performing_rank",
    round(
        top_performing_assets.avg_market_cap_rank * 0.4
        + top_performing_assets.avg_price_rank * 0.4
        + top_performing_assets.vol_market_rank * 0.2
    ),
).orderBy("top_performing_rank")

top_performing_assets.show(5)

+-----------+------------------+-------------------+-------------+--------------+---------------+----------------+---------------+-------------------+
|       name|    avg_market_cap|avg_market_cap_rank|average_price|avg_price_rank|   total_volume|vol_market_ratio|vol_market_rank|top_performing_rank|
+-----------+------------------+-------------------+-------------+--------------+---------------+----------------+---------------+-------------------+
|   Ethereum| 2.447566175596E11|                  2|     2026.751|             4|3.1391592618E10|         0.12236|             27|                8.0|
|    Bitcoin|1.3776601759463E12|                  1|      68839.2|             1| 7.194525264E10|         0.04957|             52|               11.0|
|     Solana|  4.92225953663E10|                  7|       86.283|            13|  6.426767762E9|         0.12365|             26|               13.0|
|Tether Gold|    2.8750949604E9|                 35|     5173.367|             3|   9.75957539